# Stability Analysis

Original uninjected dataset; methods: IF, LOF, MD, PCA, AE, HBOS, Pure, and Full. The primary metric is pairwise Jaccard similarity of the exact top-2% anomaly masks across five seeds.

In [ ]:
import os
# ============================================================
# Stability Analysis on the Original Uninjected Dataset
# Methods: IF, LOF, MD, PCA, AE, HBOS, Pure, Full
# Seeds: 42, 7, 123, 2024, 2025
# Primary metric: pairwise Jaccard similarity of exact top-2% masks
# ============================================================

from __future__ import annotations

from collections import OrderedDict
from itertools import combinations
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [ ]:
# ============================================================
# 1. Configuration
# ============================================================

ROOT_DIR = Path(
    os.environ.get("GIS_ANOMALY_WORKSPACE_ROOT", ".")
    r"\Outlier Paper\Code"
)

SEEDS = [42, 7, 123, 2024, 2025]
TOP_FRACTION = 0.02

BASELINE_FOLDER_PATTERN = "07_Baseline_Results_Seed_{seed}"
PROPOSED_FOLDER_PATTERN = "07_Proposed_Results_Seed_{seed}"

OUTPUT_DIR = ROOT_DIR / "07_Stability_Analysis_Results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PAIRWISE_MATRIX_DIR = OUTPUT_DIR / "Pairwise_Jaccard_Matrices"
PAIRWISE_MATRIX_DIR.mkdir(parents=True, exist_ok=True)


# Each method has several accepted candidate filenames. The first
# existing filename is used. This makes the script robust to the
# small naming differences visible in the saved baseline folders.
METHOD_CONFIG = OrderedDict(
    [
        (
            "IF",
            {
                "family": "baseline",
                "mask_candidates": [
                    "Baseline_IF_anomaly_labels.npy",
                    "IF_anomaly_labels.npy",
                ],
            },
        ),
        (
            "LOF",
            {
                "family": "baseline",
                "mask_candidates": [
                    "Baseline_LOF_anomaly_labels.npy",
                    "LOF_anomaly_labels.npy",
                ],
            },
        ),
        (
            "MD",
            {
                "family": "baseline",
                "mask_candidates": [
                    "Baseline_MD_anomaly_labels.npy",
                    "MD_anomaly_labels.npy",
                ],
            },
        ),
        (
            "PCA",
            {
                "family": "baseline",
                "mask_candidates": [
                    "Baseline_PCA_anomaly_labels.npy",
                    "PCA_anomaly_labels.npy",
                ],
            },
        ),
        (
            "AE",
            {
                "family": "baseline",
                "mask_candidates": [
                    "Baseline_AE_anomaly_labels.npy",
                    "AE_anomaly_labels.npy",
                ],
            },
        ),
        (
            "HBOS",
            {
                "family": "baseline",
                "mask_candidates": [
                    "Baseline_HBOS_anomaly_labels.npy",
                    "HBOS_anomaly_labels.npy",
                ],
            },
        ),
        (
            "Pure",
            {
                "family": "proposed",
                "mask_candidates": [
                    "Pure_top2pct_anomaly_mask.npy",
                    "pure_final_anomaly_labels.npy",
                ],
            },
        ),
        (
            "Full",
            {
                "family": "proposed",
                "mask_candidates": [
                    "Full_top2pct_anomaly_mask.npy",
                    "full_final_anomaly_labels.npy",
                ],
            },
        ),
    ]
)

METHOD_ORDER = list(METHOD_CONFIG.keys())


In [ ]:
# ============================================================
# 2. Path and array utilities
# ============================================================

def seed_folder(family: str, seed: int) -> Path:
    """Return the saved result folder for one family and seed."""

    if family == "baseline":
        return ROOT_DIR / BASELINE_FOLDER_PATTERN.format(seed=seed)

    if family == "proposed":
        return ROOT_DIR / PROPOSED_FOLDER_PATTERN.format(seed=seed)

    raise ValueError(f"Unknown method family: {family}")


def resolve_existing_file(folder: Path, candidates: list[str]) -> Path:
    """Return the first existing candidate file in a folder."""

    if not folder.exists():
        raise FileNotFoundError(
            "Required result folder was not found:\n"
            f"{folder}"
        )

    for filename in candidates:
        path = folder / filename
        if path.exists():
            return path

    raise FileNotFoundError(
        "None of the accepted files were found in:\n"
        f"{folder}\n"
        "Accepted filenames:\n"
        + "\n".join(candidates)
    )


def load_binary_mask(path: Path) -> np.ndarray:
    """Load and validate a one-dimensional binary anomaly mask."""

    loaded = np.load(path, mmap_mode="r")
    mask = np.asarray(loaded).reshape(-1)

    if mask.size == 0:
        raise ValueError(f"Empty mask: {path}")

    unique_values = np.unique(mask)
    if not set(unique_values.tolist()).issubset({0, 1, False, True}):
        raise ValueError(
            f"Mask is not binary: {path}\n"
            f"Observed values: {unique_values[:20]}"
        )

    return mask.astype(bool, copy=False)


def calculate_pair_metrics(
    first_mask: np.ndarray,
    second_mask: np.ndarray,
) -> dict[str, float | int]:
    """Calculate intersection, union, Jaccard and overlap coefficient."""

    if first_mask.shape != second_mask.shape:
        raise ValueError(
            "Cannot compare masks with different shapes: "
            f"{first_mask.shape} and {second_mask.shape}"
        )

    intersection = int(np.count_nonzero(first_mask & second_mask))
    union = int(np.count_nonzero(first_mask | second_mask))

    first_count = int(np.count_nonzero(first_mask))
    second_count = int(np.count_nonzero(second_mask))

    jaccard = intersection / union if union else np.nan
    overlap_coefficient = (
        intersection / min(first_count, second_count)
        if min(first_count, second_count) > 0
        else np.nan
    )

    return {
        "Intersection": intersection,
        "Union": union,
        "Jaccard": float(jaccard),
        "Overlap coefficient": float(overlap_coefficient),
        "Seed 1 selected": first_count,
        "Seed 2 selected": second_count,
    }


In [ ]:
# ============================================================
# 3. Validate folders and discover all mask files
# ============================================================

input_audit_rows: list[dict] = []
resolved_mask_paths: dict[str, dict[int, Path]] = {
    method: {} for method in METHOD_ORDER
}

for method, config in METHOD_CONFIG.items():
    for seed in SEEDS:
        folder = seed_folder(config["family"], seed)
        mask_path = resolve_existing_file(
            folder,
            config["mask_candidates"],
        )
        resolved_mask_paths[method][seed] = mask_path

        input_audit_rows.append(
            {
                "Method": method,
                "Family": config["family"],
                "Seed": seed,
                "Folder": str(folder.resolve()),
                "Mask file": mask_path.name,
                "Mask path": str(mask_path.resolve()),
            }
        )

input_audit_df = pd.DataFrame(input_audit_rows)

print("All result folders and anomaly-mask files were found.")
print(input_audit_df[["Method", "Seed", "Mask file"]].to_string(index=False))


In [ ]:
# ============================================================
# 4. Pairwise Jaccard and multi-seed consensus analysis
# ============================================================

pairwise_rows: list[dict] = []
summary_rows: list[dict] = []
consensus_rows: list[dict] = []
frequency_rows: list[dict] = []
validation_rows: list[dict] = []

jaccard_matrices: dict[str, pd.DataFrame] = {}
method_pair_values: dict[str, list[float]] = {}

n_samples: int | None = None
expected_selected: int | None = None

for method in METHOD_ORDER:
    print("\n" + "=" * 72)
    print(f"Processing stability for: {method}")
    print("=" * 72)

    masks: dict[int, np.ndarray] = {}
    selected_counts: dict[int, int] = {}

    # Load the five seed-specific masks for this method.
    for seed in SEEDS:
        mask_path = resolved_mask_paths[method][seed]
        mask = load_binary_mask(mask_path)

        if n_samples is None:
            n_samples = int(mask.size)
            expected_selected = int(np.ceil(TOP_FRACTION * n_samples))

        if mask.size != n_samples:
            raise ValueError(
                f"{method}, seed {seed}: mask length {mask.size:,}; "
                f"expected {n_samples:,}."
            )

        selected_count = int(mask.sum())

        if selected_count != expected_selected:
            raise ValueError(
                f"{method}, seed {seed}: selected {selected_count:,}; "
                f"expected exact top-2% count {expected_selected:,}."
            )

        masks[seed] = mask
        selected_counts[seed] = selected_count

        validation_rows.append(
            {
                "Method": method,
                "Seed": seed,
                "Mask length": int(mask.size),
                "Selected pixels": selected_count,
                "Expected selected pixels": expected_selected,
                "Exact count passed": selected_count == expected_selected,
                "Binary mask passed": True,
                "File": str(mask_path.resolve()),
            }
        )

    # Pairwise comparisons: C(5, 2) = 10 values per method.
    matrix = pd.DataFrame(
        np.eye(len(SEEDS), dtype=float),
        index=SEEDS,
        columns=SEEDS,
    )

    current_pair_values: list[float] = []

    for first_seed, second_seed in combinations(SEEDS, 2):
        metrics = calculate_pair_metrics(
            masks[first_seed],
            masks[second_seed],
        )

        matrix.loc[first_seed, second_seed] = metrics["Jaccard"]
        matrix.loc[second_seed, first_seed] = metrics["Jaccard"]
        current_pair_values.append(metrics["Jaccard"])

        pairwise_rows.append(
            {
                "Method": method,
                "Seed 1": first_seed,
                "Seed 2": second_seed,
                "Seed pair": f"{first_seed}-{second_seed}",
                **metrics,
            }
        )

    jaccard_matrices[method] = matrix
    method_pair_values[method] = current_pair_values

    pair_array = np.asarray(current_pair_values, dtype=float)

    summary_rows.append(
        {
            "Method": method,
            "Number of seeds": len(SEEDS),
            "Number of seed pairs": len(current_pair_values),
            "Mean pairwise Jaccard": float(pair_array.mean()),
            "Std. pairwise Jaccard": float(pair_array.std(ddof=1)),
            "Median pairwise Jaccard": float(np.median(pair_array)),
            "Minimum pairwise Jaccard": float(pair_array.min()),
            "Maximum pairwise Jaccard": float(pair_array.max()),
            "Mean pairwise overlap coefficient": float(
                np.mean(
                    [
                        row["Overlap coefficient"]
                        for row in pairwise_rows
                        if row["Method"] == method
                    ]
                )
            ),
            "Selected pixels per seed": expected_selected,
        }
    )

    # Multi-seed consensus: how many of the five runs selected each pixel?
    selection_frequency = np.zeros(n_samples, dtype=np.uint8)
    for seed in SEEDS:
        selection_frequency += masks[seed].astype(np.uint8, copy=False)

    frequency_counts = np.bincount(
        selection_frequency,
        minlength=len(SEEDS) + 1,
    )

    union_count = int(np.count_nonzero(selection_frequency > 0))
    all_seed_intersection = int(
        np.count_nonzero(selection_frequency == len(SEEDS))
    )
    majority_threshold = len(SEEDS) // 2 + 1
    majority_count = int(
        np.count_nonzero(selection_frequency >= majority_threshold)
    )

    multi_seed_jaccard = (
        all_seed_intersection / union_count
        if union_count > 0
        else np.nan
    )

    core_retention = (
        all_seed_intersection / expected_selected * 100.0
        if expected_selected > 0
        else np.nan
    )

    union_expansion = (
        union_count / expected_selected
        if expected_selected > 0
        else np.nan
    )

    ever_selected_frequency = selection_frequency[selection_frequency > 0]
    mean_support_ever_selected = float(
        ever_selected_frequency.mean() / len(SEEDS) * 100.0
    )

    consensus_rows.append(
        {
            "Method": method,
            "All-seed intersection": all_seed_intersection,
            "Any-seed union": union_count,
            "Multi-seed Jaccard": float(multi_seed_jaccard),
            "Core retention (%)": float(core_retention),
            "Majority threshold": majority_threshold,
            "Majority-supported pixels": majority_count,
            "Union expansion ratio": float(union_expansion),
            "Mean seed support among ever-selected pixels (%)": (
                mean_support_ever_selected
            ),
        }
    )

    for support_count in range(len(SEEDS) + 1):
        pixel_count = int(frequency_counts[support_count])
        union_share = (
            pixel_count / union_count * 100.0
            if support_count > 0 and union_count > 0
            else 0.0
        )

        frequency_rows.append(
            {
                "Method": method,
                "Number of selecting seeds": support_count,
                "Pixel count": pixel_count,
                "Dataset share (%)": pixel_count / n_samples * 100.0,
                "Share among ever-selected pixels (%)": union_share,
            }
        )

    matrix.to_csv(
        PAIRWISE_MATRIX_DIR / f"{method}_Jaccard_Matrix.csv"
    )

    # Release the large masks before loading the next method.
    del masks
    del selection_frequency


pairwise_df = pd.DataFrame(pairwise_rows)
summary_df = pd.DataFrame(summary_rows).set_index("Method").loc[METHOD_ORDER].reset_index()
consensus_df = pd.DataFrame(consensus_rows).set_index("Method").loc[METHOD_ORDER].reset_index()
frequency_df = pd.DataFrame(frequency_rows)
validation_df = pd.DataFrame(validation_rows)

summary_df = summary_df.merge(
    consensus_df,
    on="Method",
    how="left",
)


In [ ]:
# ============================================================
# 5. Wide tables for paper-oriented comparison
# ============================================================

pair_order = [f"{a}-{b}" for a, b in combinations(SEEDS, 2)]

pairwise_wide_df = pairwise_df.pivot(
    index="Method",
    columns="Seed pair",
    values="Jaccard",
).reindex(index=METHOD_ORDER, columns=pair_order)

pairwise_wide_df = pairwise_wide_df.reset_index()

frequency_union_pct_df = (
    frequency_df[frequency_df["Number of selecting seeds"] > 0]
    .pivot(
        index="Method",
        columns="Number of selecting seeds",
        values="Share among ever-selected pixels (%)",
    )
    .reindex(index=METHOD_ORDER, columns=range(1, len(SEEDS) + 1))
    .reset_index()
)

paper_summary_df = summary_df[
    [
        "Method",
        "Mean pairwise Jaccard",
        "Std. pairwise Jaccard",
        "Minimum pairwise Jaccard",
        "Maximum pairwise Jaccard",
        "All-seed intersection",
        "Core retention (%)",
        "Multi-seed Jaccard",
    ]
].copy()


In [ ]:
# ============================================================
# 6. Figures
# ============================================================

# 6A. Mean pairwise Jaccard with one-standard-deviation error bars.
plt.figure(figsize=(9, 5.5))
plt.errorbar(
    np.arange(len(summary_df)),
    summary_df["Mean pairwise Jaccard"].to_numpy(dtype=float),
    yerr=summary_df["Std. pairwise Jaccard"].to_numpy(dtype=float),
    marker="o",
    linestyle="none",
    capsize=4,
)
plt.xticks(np.arange(len(summary_df)), summary_df["Method"])
plt.ylim(0.0, 1.02)
plt.xlabel("Method")
plt.ylabel("Pairwise Jaccard similarity")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "Stability_Mean_Pairwise_Jaccard.png",
    dpi=300,
    bbox_inches="tight",
)
plt.close()


# 6B. Distribution of the ten pairwise Jaccard values per method.
plt.figure(figsize=(10, 5.5))
plt.boxplot(
    [method_pair_values[method] for method in METHOD_ORDER],
    labels=METHOD_ORDER,
    showmeans=True,
)
plt.ylim(0.0, 1.02)
plt.xlabel("Method")
plt.ylabel("Pairwise Jaccard similarity")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "Stability_Pairwise_Jaccard_Boxplot.png",
    dpi=300,
    bbox_inches="tight",
)
plt.close()


# 6C. Method-by-seed-pair Jaccard heatmap.
heatmap_values = pairwise_wide_df.set_index("Method")[pair_order].to_numpy(dtype=float)

plt.figure(figsize=(13, 5.5))
image = plt.imshow(
    heatmap_values,
    aspect="auto",
    vmin=0.0,
    vmax=1.0,
    interpolation="nearest",
)
plt.colorbar(image, label="Jaccard similarity")
plt.xticks(
    np.arange(len(pair_order)),
    pair_order,
    rotation=35,
    ha="right",
)
plt.yticks(np.arange(len(METHOD_ORDER)), METHOD_ORDER)

for row_index in range(heatmap_values.shape[0]):
    for column_index in range(heatmap_values.shape[1]):
        value = heatmap_values[row_index, column_index]
        text_color = "white" if value < 0.50 else "black"
        plt.text(
            column_index,
            row_index,
            f"{value:.3f}",
            ha="center",
            va="center",
            color=text_color,
            fontsize=8,
        )

plt.xlabel("Seed pair")
plt.ylabel("Method")
plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "Stability_Jaccard_Heatmap.png",
    dpi=300,
    bbox_inches="tight",
)
plt.close()


# 6D. Pixels retained in the anomaly set under all five seeds.
plt.figure(figsize=(9, 5.5))
plt.bar(
    np.arange(len(summary_df)),
    summary_df["Core retention (%)"].to_numpy(dtype=float),
)
plt.xticks(np.arange(len(summary_df)), summary_df["Method"])
plt.ylim(0.0, 102.0)
plt.xlabel("Method")
plt.ylabel("Selected pixels retained across all seeds (%)")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "Stability_All_Seed_Core_Retention.png",
    dpi=300,
    bbox_inches="tight",
)
plt.close()


# 6E. Seed-support distribution among pixels selected at least once.
frequency_values = frequency_union_pct_df.set_index("Method")[
    list(range(1, len(SEEDS) + 1))
].to_numpy(dtype=float)

plt.figure(figsize=(9, 5.5))
image = plt.imshow(
    frequency_values,
    aspect="auto",
    vmin=0.0,
    vmax=100.0,
    interpolation="nearest",
)
plt.colorbar(image, label="Share among ever-selected pixels (%)")
plt.xticks(
    np.arange(len(SEEDS)),
    [f"{support}/{len(SEEDS)}" for support in range(1, len(SEEDS) + 1)],
)
plt.yticks(np.arange(len(METHOD_ORDER)), METHOD_ORDER)

for row_index in range(frequency_values.shape[0]):
    for column_index in range(frequency_values.shape[1]):
        value = frequency_values[row_index, column_index]
        text_color = "white" if value < 50.0 else "black"
        plt.text(
            column_index,
            row_index,
            f"{value:.1f}",
            ha="center",
            va="center",
            color=text_color,
            fontsize=8,
        )

plt.xlabel("Number of seeds selecting the pixel")
plt.ylabel("Method")
plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "Stability_Seed_Support_Heatmap.png",
    dpi=300,
    bbox_inches="tight",
)
plt.close()


In [ ]:
# ============================================================
# 7. Save CSV and LaTeX tables
# ============================================================

input_audit_df.to_csv(
    OUTPUT_DIR / "Stability_Input_Audit.csv",
    index=False,
)

validation_df.to_csv(
    OUTPUT_DIR / "Stability_Validation.csv",
    index=False,
)

pairwise_df.to_csv(
    OUTPUT_DIR / "Stability_Pairwise_Jaccard_Long.csv",
    index=False,
)

pairwise_wide_df.to_csv(
    OUTPUT_DIR / "Stability_Pairwise_Jaccard_Wide.csv",
    index=False,
)

summary_df.to_csv(
    OUTPUT_DIR / "Stability_Method_Summary.csv",
    index=False,
)

consensus_df.to_csv(
    OUTPUT_DIR / "Stability_Consensus_Summary.csv",
    index=False,
)

frequency_df.to_csv(
    OUTPUT_DIR / "Stability_Selection_Frequency.csv",
    index=False,
)

frequency_union_pct_df.to_csv(
    OUTPUT_DIR / "Stability_Seed_Support_Among_Selected.csv",
    index=False,
)

with open(
    OUTPUT_DIR / "Stability_Method_Summary.tex",
    "w",
    encoding="utf-8",
) as latex_file:
    latex_file.write(
        paper_summary_df.to_latex(
            index=False,
            float_format="%.4f",
            escape=True,
        )
    )


In [ ]:
# ============================================================
# 8. Save the complete Excel workbook
# ============================================================

excel_output_path = OUTPUT_DIR / "Stability_Analysis_Results.xlsx"

# Prefer XlsxWriter for a newly created workbook, with an automatic
# fallback to openpyxl when XlsxWriter is unavailable.
try:
    import xlsxwriter  # noqa: F401
    excel_engine = "xlsxwriter"
except ImportError:
    excel_engine = "openpyxl"

with pd.ExcelWriter(excel_output_path, engine=excel_engine) as writer:
    paper_summary_df.to_excel(writer, sheet_name="Paper Summary", index=False)
    summary_df.to_excel(writer, sheet_name="Full Summary", index=False)
    pairwise_df.to_excel(writer, sheet_name="Pairwise Long", index=False)
    pairwise_wide_df.to_excel(writer, sheet_name="Pairwise Wide", index=False)
    consensus_df.to_excel(writer, sheet_name="Consensus", index=False)
    frequency_df.to_excel(writer, sheet_name="Frequency Long", index=False)
    frequency_union_pct_df.to_excel(
        writer,
        sheet_name="Seed Support",
        index=False,
    )
    validation_df.to_excel(writer, sheet_name="Validation", index=False)
    input_audit_df.to_excel(writer, sheet_name="Input Audit", index=False)

    for method, matrix in jaccard_matrices.items():
        matrix.to_excel(
            writer,
            sheet_name=f"J_{method}",
            index=True,
        )

    if excel_engine == "xlsxwriter":
        workbook = writer.book
        header_format = workbook.add_format(
            {
                "bold": True,
                "text_wrap": True,
                "valign": "top",
                "border": 1,
            }
        )
        decimal_format = workbook.add_format({"num_format": "0.0000"})
        percent_format = workbook.add_format({"num_format": "0.00"})

        for sheet_name, worksheet in writer.sheets.items():
            worksheet.freeze_panes(1, 0)
            worksheet.autofilter(0, 0, 0, max(0, worksheet.dim_colmax))
            worksheet.set_row(0, 32, header_format)
            worksheet.set_column(0, 0, 18)
            worksheet.set_column(1, max(1, worksheet.dim_colmax), 18)

        writer.sheets["Paper Summary"].set_column(1, 4, 20, decimal_format)
        writer.sheets["Paper Summary"].set_column(6, 6, 18, percent_format)


In [ ]:
# ============================================================
# 9. Save manifest and analysis notes
# ============================================================

manifest = {
    "analysis": "model-seed stability on original uninjected dataset",
    "seeds": SEEDS,
    "methods": METHOD_ORDER,
    "number_of_samples": int(n_samples),
    "top_fraction": TOP_FRACTION,
    "selected_pixels_per_method_per_seed": int(expected_selected),
    "number_of_pairwise_comparisons_per_method": len(pair_order),
    "primary_metric": "pairwise Jaccard similarity of exact top-2% masks",
    "secondary_metrics": [
        "pairwise overlap coefficient",
        "all-seed intersection",
        "any-seed union",
        "multi-seed Jaccard",
        "core retention",
        "selection-frequency distribution",
    ],
    "baseline_folder_pattern": BASELINE_FOLDER_PATTERN,
    "proposed_folder_pattern": PROPOSED_FOLDER_PATTERN,
    "output_directory": str(OUTPUT_DIR.resolve()),
}

with open(
    OUTPUT_DIR / "stability_analysis_manifest.json",
    "w",
    encoding="utf-8",
) as manifest_file:
    json.dump(manifest, manifest_file, indent=4)

notes_text = f"""STABILITY ANALYSIS SCOPE
========================

Dataset:
Original uninjected dataset with {n_samples:,} valid pixels.

Methods:
{', '.join(METHOD_ORDER)}

Seeds:
{', '.join(str(seed) for seed in SEEDS)}

Selection rule:
Each method independently selects exactly the highest-ranked top {TOP_FRACTION * 100:.0f}%
of pixels in each seed run, corresponding to {expected_selected:,} pixels.

Primary metric:
Pairwise Jaccard similarity between exact top-2% anomaly masks. With five
seeds, each method has ten pairwise comparisons.

Interpretation:
Higher pairwise Jaccard indicates that the method identifies a more consistent
anomaly population when its training, fitting subset, initialization, and other
seed-controlled operations change. The analysis measures reproducibility of the
selected anomaly set; it does not measure detection accuracy because complete
ground truth is unavailable for naturally occurring anomalies in the original
dataset.

All-seed core:
The all-seed intersection is the number of pixels selected by every one of the
five runs. Core retention divides this intersection by the fixed top-2% set size.

Multi-seed Jaccard:
Intersection of all five masks divided by their union.
"""

with open(
    OUTPUT_DIR / "README_Stability_Analysis.txt",
    "w",
    encoding="utf-8",
) as notes_file:
    notes_file.write(notes_text)


In [ ]:
# ============================================================
# 10. Final validation and console summary
# ============================================================

if not validation_df["Exact count passed"].all():
    raise RuntimeError("At least one exact top-2% count validation failed.")

if pairwise_df["Jaccard"].isna().any():
    raise RuntimeError("NaN Jaccard value found.")

if not pairwise_df["Jaccard"].between(0.0, 1.0).all():
    raise RuntimeError("A Jaccard value lies outside [0, 1].")

required_outputs = [
    "Stability_Analysis_Results.xlsx",
    "Stability_Method_Summary.csv",
    "Stability_Pairwise_Jaccard_Long.csv",
    "Stability_Mean_Pairwise_Jaccard.png",
    "Stability_Pairwise_Jaccard_Boxplot.png",
    "Stability_Jaccard_Heatmap.png",
    "Stability_All_Seed_Core_Retention.png",
    "Stability_Seed_Support_Heatmap.png",
    "stability_analysis_manifest.json",
    "README_Stability_Analysis.txt",
]

missing_outputs = [
    filename
    for filename in required_outputs
    if not (OUTPUT_DIR / filename).exists()
]

if missing_outputs:
    raise FileNotFoundError(
        "Stability analysis completed incompletely. Missing outputs:\n"
        + "\n".join(missing_outputs)
    )

print("\n" + "=" * 72)
print("STABILITY ANALYSIS COMPLETED")
print("=" * 72)
print(f"Samples: {n_samples:,}")
print(f"Exact selected pixels per run: {expected_selected:,}")
print(f"Seeds: {SEEDS}")
print(f"Methods: {METHOD_ORDER}")
print("\nMethod summary:")
print(
    summary_df[
        [
            "Method",
            "Mean pairwise Jaccard",
            "Std. pairwise Jaccard",
            "Minimum pairwise Jaccard",
            "Maximum pairwise Jaccard",
            "Core retention (%)",
        ]
    ]
    .round(4)
    .to_string(index=False)
)
print("\nAll outputs saved in:")
print(OUTPUT_DIR.resolve())
